In [ ]:
# --- Act 0: Setup (hidden) ---
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')

# The user IPython profile stubs sys.modules['kaleido']=None on ARM/Tegra as an
# old kaleido-0.x SIGABRT guard.  kaleido 1.x renders static PNGs via
# Chrome/choreographer and does not import tensorflow/jax, so un-stub it here so
# the plotly charts export under nbconvert regardless of the active profile.
for _stubbed in ('kaleido', 'kaleido.scopes', 'kaleido.scopes.plotly'):
    if _stubbed in sys.modules and sys.modules[_stubbed] is None:
        del sys.modules[_stubbed]
try:  # reset plotly's cached availability probe if it ran before the un-stub
    import plotly.io._kaleido as _pk
    _pk._KALEIDO_AVAILABLE = None
    _pk._KALEIDO_MAJOR = None
except Exception:
    pass

import html
import json
import re
from collections import Counter
from pathlib import Path
from textwrap import shorten

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from scipy import stats


# --- CWD-robust repository-root resolution ---
# The preprint build executes a COPY of this notebook with CWD =
# notebooks/preprint/, so REPO_ROOT must not depend on the CWD depth.  Walk up
# from the current directory to the repo marker (pyproject.toml + engine/); fall
# back to `git rev-parse --show-toplevel`.  Never use a fixed relative depth.
def _find_repo_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / 'pyproject.toml').exists() and (base / 'engine').is_dir():
            return base
    try:
        top = subprocess.check_output(
            ['git', 'rev-parse', '--show-toplevel'], text=True
        ).strip()
        if top:
            return Path(top)
    except Exception:
        pass
    raise FileNotFoundError(
        "Cannot locate repository root (no pyproject.toml with engine/ walking "
        "up from CWD, and `git rev-parse --show-toplevel` failed)."
    )

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026'
RARR_CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026-rarr'
BASELINES = REPO_ROOT / 'projects' / 'owasp-llm' / 'baselines' / '2026'

# --- Preprint figure output dir (300-dpi PNGs saved by the chart library) ---
PREPRINT_FIG = REPO_ROOT / 'notebooks' / 'preprint' / 'figures'
PREPRINT_FIG.mkdir(parents=True, exist_ok=True)

# --- Tested data + chart library (remediation #1: one tested chart library) ---
from engine.report.blend_2025_2026 import (
    blended_ranking,
    load_entries,
    rank_moves,
)
from engine.report.narrative_charts import (
    render_bump_chart,
    render_ci_overlap,
    render_confusion_heatmap,
    render_confusion_matrix_3x3,
    render_dumbbell_chart,
    render_entry_expansion_map,
    render_oos_treemap,
    render_paired_dots,
    render_precision_bars,
    render_precision_posteriors,
    render_rarr_robustness,
    render_ridge_plot,
    render_sankey_confusion,
    render_stratum_bar,
    render_theme_bars,
    render_tier_donut,
)
from engine.report.narrative_data import load_narrative_data

# --- Seaborn / matplotlib theme ---
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

FRAME_BLIND = {'LLM04', 'LLM08', 'LLM10'}

# --- Deep-dive sidebar helpers (used by the narrative markdown/HTML cells) ---
def sidebar(title, content):
    """Render a collapsible deep-dive sidebar."""
    return HTML(
        f'<details style="margin: 1em 0; padding: 0.5em; '
        f'border-left: 3px solid #4a86c8; background: #f8f9fa;">'
        f'<summary style="cursor: pointer; font-weight: bold; '
        f'color: #2c5282;">{title}</summary>'
        f'<div style="margin-top: 0.8em; line-height: 1.6;">{content}</div>'
        f'</details>'
    )

def callout_box(text, border_color='#e53e3e', bg_color='#fff5f5'):
    """Render an always-visible boxed callout (not collapsed)."""
    return HTML(
        f'<div style="margin: 1em 0; padding: 1em; border-left: 4px solid {border_color}; '
        f'background: {bg_color}; line-height: 1.6;">{text}</div>'
    )

# --- Load all pre-computed data via the tested loader ---
DATA = load_narrative_data(CYCLE)
ENTRY_NAMES = DATA['entry_names']
INFER_ENTRY_ORDER = DATA['entry_ids']

# Validate shapes
assert DATA['lambda_samples'].shape == (16000, 20), (
    f"Expected lambda_samples shape (16000, 20), got {DATA['lambda_samples'].shape}"
)
assert len(INFER_ENTRY_ORDER) == 20, (
    f"Expected 20 entry_ids in inference_summary, got {len(INFER_ENTRY_ORDER)}"
)

print(f"Loaded data via load_narrative_data. {len(DATA['incidents'])} incidents, "
      f"{len(DATA['prelabels'])} prelabels, {len(DATA['goldset'])} adjudications, "
      f"{DATA['lambda_samples'].shape[0]} posterior draws. "
      f"Figures -> {PREPRINT_FIG}")


# How to Read This Report

This report checks the 2026 OWASP Top 10 for LLM Applications against a record of real incidents, and it is written for a security professional who has never taken a statistics course. The argument runs top to bottom in plain language. Wherever a data-science term first appears, a short sidebar box defines it:

> **Sidebar — example.** Sidebars look like this. Each one explains a single term in plain language. Skip them when the term is already familiar; the main argument reads without them.

Every term defined in a sidebar reappears in the Glossary at the end, so this doubles as a reference.

**Scope, in one line.** This is an incident-data analysis by two working-group members. It is not the official OWASP release and does not set or supersede the official list. The full scope statement is at the end.

The structure:

- **Part I** explains what the OWASP LLM Top 10 is, where the incident corpus comes from, and how the expert vote and the data combine into one ranking.
- **Part II** walks through what the incident data says, one step at a time: the corpus, the classifier, its measured accuracy, the Bayesian model, the incident-derived ranking, and where that ranking agrees and disagrees with the experts.
- **Part III** stress-tests the ranking against four frontier classifiers and a ground-truth check.
- **Limitations**, **Glossary**, and **Scope** close the report.

# Part I — The List and How It's Made

## What the list is, and why check it against incidents

The OWASP Top 10 for LLM Applications is a ranked list of the security risks that matter most when software is built on large language models. It is an expert-consensus product: practitioners score each candidate risk, and the scores aggregate into an ordered list, from Prompt Injection at the top down to the tenth entry. Organizations use the list to steer defensive effort: which risks get a control, which get a test case, which get an audit line.

An expert vote is one way to rank risk. It captures informed judgment, but judgment carries its own biases: what practitioners have read about recently, and what last year's list anchored them to. We wanted to check the vote against a second, independent signal, the pattern of incidents that have actually happened. A risk that is common in the field should leave a trail in public incident records. Comparing the two signals is the whole of this report, and the result is a calibrated one: the incident data agrees with the expert ranking only weakly (Cohen's κ ≈ 0.20, with an interval that crosses zero), and the expert ranking is robust: four frontier classifiers and a ground-truth check do not move it.

> **Sidebar — incident corpus.** A corpus is a fixed, documented collection of records assembled for analysis. Our incident corpus is a snapshot of publicly reported LLM-security incidents, each one a short text description of something that went wrong, pulled from public databases on a fixed date so the analysis reproduces exactly.

## Where the corpus comes from

The corpus holds 7,714 incidents as snapshotted, of which 6,639 carry a usable label against the taxonomy. Those 6,639 divide into two strata: 6,297 security incidents and 342 ai-harm incidents. The security stratum comes from three public vulnerability databases (CVE, GHSA, and OSV) and reads like engineering: advisory and patch text for exploits and data-leakage bugs. The ai-harm stratum comes from AIAAIC, a database of AI-related harms and controversies, and reads like journalism: news summaries of algorithmic discrimination and deepfake misuse.

A second corpus corroborates the first at smaller scale. The OWASP Agentic Security Initiative contributed 46 independently curated agentic-AI incidents; classifying them against the same taxonomy reproduced 26% of the label assignments. That is a modest agreement rate on a small, differently sourced set, and we report it as corroboration, not proof. We describe the main corpus as large-scale (7,714 incidents is a large sample for this field) and make no claim to being first.

## The 0.75 / 0.25 blend

The 2026 list comes from neither signal alone. It comes from a weighted blend of the two. The expert signal is a practitioner survey: about 29 respondents scored each candidate risk on importance, and the scores aggregate to a median rank per entry. The data signal is the incident-derived rank that Part II builds. The blend combines them at fixed weights.

The two witnesses are combined in score space. Each risk's expert rank and incident rate are put on a common scale, weighted three-quarters to the vote and one quarter to the data, and blended per posterior draw. On the data side the rates of a rolled-up child add to their parent, since incidents accumulate. Three risks whose incident recall the corpus cannot estimate — Data and Model Poisoning, Vector and Embedding Weaknesses, and Unbounded Consumption — take their position from the vote alone. The result is a distribution over positions, reported as three tiers in the final section.

The weighting is deliberately lopsided, for a reason this report builds toward. The list is a practitioner-consensus artifact, so the consensus leads. The incident data is a useful but noisy corrective: it under-detects unevenly, it is drawn from whatever reaches public databases, and it agrees with the expert ranking only weakly. At a quarter weight, the data is strong enough to move a risk a full tier when the gap between the two signals is large, and too weak to overturn the consensus on one imperfect corpus. That balance is the point of the split: the data tugs, the consensus holds.

## What changed from 2025 to 2026

The 2026 cycle works from a field of 20 candidate entries, not 10. Ten are incumbents carried over from 2025 (LLM01 through LLM10). Six are new candidates the working group is tracking but has held off the published list (the NEW-* entries). Four are narrower risks the group folded into a broader incumbent rather than list separately (the ROLL-* rollups); the incidents belonging to a rolled-up child still count, now toward its parent.

![The 2026 candidate field: ten incumbents, six new candidates, and four rollups, each rollup arrowed to the incumbent it folds into.](figures/entry_expansion_map.png){width=85%}

Blending the two signals reorders the ten incumbents from their published 2025 positions, and the reordering settles into the three tiers this report uses throughout: a co-leading pair, a tied middle band, and a wide tail. Excessive Agency is the one entry with a clear stated move, entering the tied band. Every other incumbent's position shifts inside its tier or holds; the model reports that movement as a spread of plausible ranks per risk. The position chart in the blended Top-10 section shows every incumbent's tier and its range of plausible placements.

# Part II — What the Incident Data Says

## Act 1: The question

The 2025 OWASP Top 10 for LLM Applications came from expert consensus: practitioner votes aggregated into a ranking, Prompt Injection at #1, Sensitive Information Disclosure at #2, on down to #10. This part checks that consensus against a second signal, the incident record, one step at a time.

We built a corpus of 6,639 labeled LLM-security incidents from public databases, classified each against the 20-entry taxonomy, and derived a data-driven ranking. The question is not whether the data proves the experts right — it cannot, and Part III shows the agreement is weak. The question is what the data says on its own, and where it lines up with the vote and where it does not.

Every chart and table below is computed live from the committed data; re-run any cell to check it. The walkthrough covers how the classification worked, how we measured its accuracy, and what a Bayesian model does with noisy measurements. Each data-science term gets a sidebar on first use.

The 20 taxonomy entries are listed below. The "Incident Rank" column is blank for now; we fill it in Act 6, after the methodology.

In [ ]:
# Build the entry table with a placeholder rank column
entry_table = pd.DataFrame([
    {'#': i + 1, 'Entry ID': e['entry_id'], 'Name': e['canonical_name'], 'Incident Rank': '—'}
    for i, e in enumerate(DATA['rubric']['entries'])
])
entry_table = entry_table.set_index('#')
display(entry_table.style.set_caption(
    "20 taxonomy entries. Incident Rank will be filled in Act 6."
).set_properties(**{'text-align': 'left'}))

## Act 2: The corpus

The corpus holds 6,639 labeled incidents in two strata that read very differently. The security stratum (6,297 incidents from CVE, GHSA, and OSV) is written like engineering: advisory text and patch notes for prompt-injection exploits and data leakage through APIs. The ai-harm stratum (342 incidents from AIAAIC) is written like journalism: news summaries of algorithmic discrimination and deepfake misuse.

The split matters for what follows. The classifier reads the two strata differently, and, as Act 4 shows, we could hand-verify its precision only on the security stratum. Counts and corrections for ai-harm categories therefore rest on weaker measurement than those for security categories.

![Labeled incidents by stratum: the security sources (CVE, GHSA, OSV) against the ai-harm source (AIAAIC).](figures/stratum_bar.png){width=42% wrap=right}

In [ ]:
# Act 2: incidents-by-stratum bar chart (saves stratum_bar.png @ 300 dpi)
render_stratum_bar(DATA, PREPRINT_FIG)


In [ ]:
# Show 2 real incident examples — one from each stratum
# Build stratum lookup from classified incidents (prelabels has no stratum field)
_stratum_lookup = {inc['incident_id']: inc['stratum'] for inc in DATA['incidents']}

prelabels_df = DATA['prelabels']
security_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                    and p['consensus'] not in ('out-of-scope', None)
                    and _stratum_lookup.get(p['incident_id']) == 'security'), None)
harm_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                and _stratum_lookup.get(p['incident_id']) == 'ai-harm'
                and len(p['text']) > 100), None)
if security_ex is None or harm_ex is None:
    display(HTML('<p style="color: red;">Could not find suitable example incidents.</p>'))

def incident_card(record, label):
    """Format an incident as an HTML card."""
    text = html.escape(shorten(record['text'], width=250, placeholder='...'))
    consensus = html.escape(str(record['consensus']))
    tier = html.escape(str(record['triage_tier']))
    inc_id = html.escape(str(record['incident_id']))
    return HTML(
        f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #fafafa;">'
        f'<strong>{html.escape(label)}</strong><br>'
        f'<code>{inc_id}</code> · consensus: '
        f'<strong>{consensus}</strong> · tier: {tier}<br>'
        f'<p style="margin-top: 0.5em; color: #333;">{text}</p>'
        f'</div>'
    )

display(HTML('<h4>Example: Security stratum (CVE/GHSA/OSV)</h4>'))
display(incident_card(security_ex, 'Security incident'))

display(HTML('<h4>Example: AI-harm stratum (AIAAIC)</h4>'))
display(incident_card(harm_ex, 'AI-harm incident'))

In [ ]:
# F-frame sidebar
display(sidebar(
    'Deep dive: What the corpus cannot see (F-frame)',
    '<p>The corpus is built from a keyword crawl of public databases — CVE, GHSA, OSV, '
    'and AIAAIC. Incidents that never became CVEs or harm-database entries are invisible '
    'to us. A prompt injection attack against an internal enterprise tool that was caught '
    'and patched quietly will never appear in this data.</p>'
    '<p>This creates structural bias toward vulnerability types that get reported in '
    'public channels. Well-resourced organizations that fix issues internally are '
    'underrepresented. Novel attack types that have not been assigned a CVE category '
    'are invisible.</p>'
))

# F-circ callout — always visible, not collapsed
display(callout_box(
    '<strong>Structural limitation: taxonomy-frame circularity (F-circ)</strong><br><br>'
    'We classified these incidents using the same taxonomy we are trying to validate. '
    'If the classifier systematically favors certain entries, the incident counts will '
    'appear to confirm the expert rankings even if the true pattern is different. '
    'This is taxonomy-frame circularity. It means the concordance we measure later '
    'is an upper bound on true agreement, not a precise estimate of it.',
    border_color='#d69e2e', bg_color='#fefcbf'
))

## Act 3: Classifying 6,639 incidents

Three large language models classified each incident independently: Qwen 235B, Llama 405B, and DeepSeek V3. Each read the incident text and assigned it to one of the 20 taxonomy entries, or marked it out of scope when none fit.

> **Sidebar — classifier.** A classifier is any procedure that reads an input and assigns it to one of a fixed set of categories. Here the input is an incident's text and the categories are the 20 taxonomy entries plus "out of scope." Our classifier is an ensemble of three language models voting. It is a measuring instrument, not ground truth, which is why Act 4 measures how often it is right.

> **Sidebar — out-of-scope.** "Out of scope" is the category for incidents that belong to none of the 20 taxonomy entries — a real AI harm that is not a vulnerability in a large language model, such as a biased hiring tool or a surveillance drone. Marking an incident out of scope is a correct classification, not a failure to classify.

When all three models agreed on the same entry, we call it the agree tier. When two agreed and one differed, the split tier. When all three picked different entries, the disagree tier. The tier is a confidence signal: agree-tier incidents have strong three-model consensus; disagree-tier incidents sit in ambiguous territory where three independent classifiers could not converge.

![Consensus tiers across the corpus: agree, split, and disagree.](figures/tier_donut.png){width=40% wrap=right}

![Where the three classifiers disagree, by entry pair, across the taxonomy.](figures/confusion_heatmap.png){width=72%}

In [ ]:
# Find a good agree-tier example with three matching votes
agree_ex = next((p for p in DATA['prelabels']
                 if p['triage_tier'] == 'agree'
                 and p['consensus'] not in ('out-of-scope', None)
                 and len(p['model_votes']) == 3
                 and len(p['text']) > 120), None)
assert agree_ex is not None, "No suitable agree-tier example found in prelabels"

display(HTML('<h4>Worked example: three-model classification</h4>'))
display(HTML(
    f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
    f'margin: 0.5em 0; background: #f7fafc;">'
    f'<p style="color: #555;"><strong>Incident text</strong> (truncated):</p>'
    f'<p style="font-style: italic;">{html.escape(shorten(agree_ex["text"], 250, placeholder="..."))}</p>'
    f'<hr style="border: 0; border-top: 1px solid #e2e8f0;">'
    f'<table style="width: 100%; border-collapse: collapse;">'
    f'<tr style="background: #edf2f7;"><th>Model</th><th>Entry</th><th>Confidence</th></tr>'
    + ''.join(
        f'<tr><td>{html.escape(v["model_id"].split("/")[-1])}</td>'
        f'<td><strong>{html.escape(v["entry_id"])}</strong></td>'
        f'<td>{v["confidence"]:.0%}</td></tr>'
        for v in agree_ex['model_votes']
    )
    + f'</table>'
    f'<p style="margin-top: 0.5em;">Consensus: <strong>{html.escape(str(agree_ex["consensus"]))}</strong> '
    f'(tier: {html.escape(str(agree_ex["triage_tier"]))})</p>'
    f'</div>'
))

In [ ]:
# Act 3: consensus-tier donut (saves tier_donut.png @ 300 dpi)
render_tier_donut(DATA, PREPRINT_FIG)


In [ ]:
# Act 3: entry-pair disagreement heatmap (saves confusion_heatmap.png @ 300 dpi)
render_confusion_heatmap(DATA, PREPRINT_FIG)


In [ ]:
display(sidebar(
    'Deep dive: Why three models?',
    '<p>Single-model classification had lower precision in our early experiments. '
    'A single model might confidently assign an incident to the wrong entry because '
    'of biases in its training data or the phrasing of the prompt.</p>'
    '<p>Three models with majority vote reduces noise the same way a panel of three '
    'judges reduces individual bias. If two of three models agree, we have higher '
    'confidence in the label. The disagree tier (where all three pick different entries) '
    'explicitly marks incidents where no classifier consensus exists.</p>'
))

display(sidebar(
    'Deep dive: The two-stage classification pipeline',
    '<p><strong>Stage 1 (heuristic):</strong> Regex and keyword indicators scan the '
    'incident text for known patterns (e.g., "prompt injection," "CVE-2026-*"). '
    'This produces a fast initial assignment at low confidence (10%). All incidents '
    'proceed to Stage 2 regardless of Stage 1 results.</p>'
    '<p><strong>Stage 2 (LLM):</strong> Each of three models reads the full incident '
    'text alongside the complete rubric (all 20 entries with inclusion/exclusion '
    'criteria). The model returns an entry_id, a confidence score, and a rationale. '
    'The three Stage 2 labels are combined into the consensus and triage tier.</p>'
))

## Act 4: How good is the classifier?

The classifier is a measuring instrument, so we measured it. Two numbers matter: how often its labels are correct, and how many true cases it finds.

> **Sidebar — precision.** Of the incidents a classifier files under a category, precision is the fraction that truly belong there. If it labels 100 incidents "LLM08" and 13 of them really are LLM08, its precision on LLM08 is 13%. Low precision means most of what lands in a category does not belong to it.

> **Sidebar — recall.** Of the incidents that truly belong to a category, recall is the fraction the classifier actually finds. A classifier can have high precision and low recall (right when it fires, but it rarely fires) or the reverse. Precision and recall answer different questions and are measured separately.

> **Sidebar — gold set.** A gold set is a batch of records labeled carefully by a human to serve as the answer key. We use it two ways: to measure the classifier's precision and recall, and, in Part III, as the ground truth a ranking is scored against. Ours holds 1,200 human-adjudicated incidents.

> **Sidebar — blind labeling.** Blind labeling means the human records an independent judgment before seeing the machine's answer, so the human is not anchored to it. Our reviewer labeled each incident from its text alone, then revealed the three model votes, then made a final decision. The blind step keeps the gold set from silently inheriting the classifier's mistakes.

We hand-verified 323 classifications as part of measuring precision, and a reviewer adjudicated 1,200 incidents across all tiers to measure recall. The precision posteriors combine those hand-verified checks with the goldset adjudications. Precision varies sharply across entries, from 93% (LLM01, LLM03) down to 13% (LLM08). The variation is not random; it tracks how cleanly each entry's definition separates it from its neighbors. Four entries fall below the 50% mark, each for a specific reason.

- **LLM08 (Vector and Embedding Weaknesses), 13%.** The lowest in the taxonomy: for every eight incidents labeled LLM08, about one belongs there. The category covers a narrow class of attacks on embedding spaces and vector stores, and the classifier confuses it with data-and-model-poisoning incidents (LLM04) and general data-integrity issues that are conceptually adjacent but taxonomically distinct.
- **LLM07 (Hidden Context Exposure), 31%.** The classifier struggles to separate exposing hidden system context (LLM07) from overriding it through injection (LLM01). Many real incidents involve both, because an attacker exposes the hidden context in order to craft a better injection. The boundary is clear in the taxonomy and blurred in practice.
- **ROLL-CFAS (Compositional Fine-tuning Alignment Subversion), 33%.** Only one precision observation, so the posterior is dominated by its Beta(1,1) prior and the 90% interval runs from 3% to 78%. The estimate says almost nothing; it reflects how little was measured, not a property of the category.
- **ROLL-CMSB (Cross-Modal Safety Bypass), 44%.** This entry sits on the confusion boundary examined in Act 9B. A deepfake that bypasses a content filter could read as cross-modal bypass (ROLL-CMSB), misinformation (LLM09), or weaponized abuse (NEW-WLA); the classifier picks one where a human might reasonably pick another.

Precision below 50% means the classifier is wrong more often than right for that entry. When you see an incident labeled LLM08, the odds are about seven-to-one against it truly being a vector-or-embedding weakness. This feeds directly into the Bayesian model in Act 5: low-precision entries get large upward corrections, because much of their observed count is misclassification noise, and wide uncertainty ranges, because the correction itself is uncertain. A 13% precision estimate does not mean the entry is unimportant. It means the automated measurement of that entry is unreliable, and the model's uncertainty says so.

One coverage caveat. All 323 precision checks came from the security stratum. The ai-harm stratum has no precision measurements, so the Bayesian model uses a flat Beta(1,1) prior (prior mean 0.5) for ai-harm precision. Error correction for ai-harm incidents rests on that uninformative prior, not on direct measurement.

![Classifier precision by entry, with the 50% line marked; precision ranges from 93% down to 13%.](figures/precision_bars.png){width=60%}

![Beta posteriors for per-entry precision; prior-dominated entries show wide, flat curves.](figures/precision_posteriors.png){width=62%}

In [ ]:
# Act 4: classifier precision bars (saves precision_bars.png @ 300 dpi)
render_precision_bars(DATA, PREPRINT_FIG)


In [ ]:
# Act 4: precision Beta posteriors (saves precision_posteriors.png @ 300 dpi)
render_precision_posteriors(DATA, PREPRINT_FIG)


In [ ]:
# precision_data was defined in the Act 4 chart cell (now a library call);
# recompute it here so this sidebar cell is self-contained.
precision_data = DATA['posteriors']['precision']

display(sidebar(
    'Deep dive: The gold-set process — 1,200 human adjudications',
    '<p>A human reviewer (the project author) adjudicated 1,200 incidents using a '
    'blind-first protocol. For each incident:</p>'
    '<ol>'
    '<li>Read the incident text without seeing the model votes (blind label).</li>'
    '<li>Record an independent classification.</li>'
    '<li>Then reveal the three model votes and the consensus.</li>'
    '<li>Make a final decision: accept the consensus, override to a different entry, '
    'assign multiple labels, or mark as out of scope.</li>'
    '</ol>'
    '<p>"Adjudication" is different from voting. The reviewer is not adding a fourth '
    'opinion — they are making a judgment call after seeing both the text and the '
    'model reasoning. The blind label (step 2) guards against anchoring to the '
    'model consensus.</p>'
))

# Full precision posteriors table
prec_table_rows = []
for key, params in sorted(precision_data.items()):
    entry_id = key.split('::')[0]
    if entry_id == 'out-of-scope':
        continue
    alpha, beta_p = params['alpha'], params['beta']
    mean = alpha / (alpha + beta_p)
    ci_low, ci_high = stats.beta.ppf([0.05, 0.95], alpha, beta_p)
    n = int(alpha + beta_p - 2)
    flag = '(prior-dominated)' if n < 5 else ''
    prec_table_rows.append({
        'Entry': entry_id,
        'alpha': alpha, 'beta': beta_p,
        'Mean': f'{mean:.1%}',
        '90% CI': f'[{ci_low:.1%}, {ci_high:.1%}]',
        'n': n,
        'Note': flag,
    })

prec_table_html = pd.DataFrame(prec_table_rows).to_html(index=False, escape=False)
display(sidebar('Deep dive: Full precision posteriors table', prec_table_html))

## Act 5: From counts to rankings — the Bayesian model

Raw incident counts would mislead. An entry whose classifier runs at 30% precision looks busy, but two-thirds of what lands there was misclassified from somewhere else. We need a model that adjusts each entry's observed count for its known classifier error and carries the uncertainty of that adjustment through to the result.

The idea is a bathroom scale that reads two pounds heavy: you subtract two pounds from every reading, and if the scale itself is uncertain (two pounds off, give or take one) the corrected weight is uncertain too. The model does this per entry, using the measured precision and recall from Act 4.

> **Sidebar — latent incidence (λ).** Latent incidence, written λ (lambda), is the quantity the model is really after: the true underlying rate at which incidents of a category occur, as opposed to the raw count the classifier reported. "Latent" means unobserved: we never see λ directly, and infer it from the noisy counts after correcting for precision and recall.

> **Sidebar — prior and posterior.** A prior is what the model assumes about a quantity before looking at this data; the posterior is the updated belief after combining the prior with the data. The posterior is not a single number but a distribution, a range of plausible values with more weight on the likelier ones. A wide posterior means the data left the answer uncertain.

> **Sidebar — negative-binomial measurement-error model.** Two ideas in one name. "Measurement-error model" means the model treats the classifier's counts as noisy measurements of the true rate and corrects for the noise. "Negative-binomial" is the count distribution it uses; unlike the simpler Poisson, it lets the spread of counts exceed their average, which real incident counts do. Together: a count model that expects over-dispersed data and corrects for classifier error.

For entries above 50% precision, the correction is a moderate downward nudge: some observed incidents were misclassified in, so the true count is a little lower. For entries below 50% precision, the correction is larger than the reading itself: if only 13% of LLM08 labels are real, the model must recover the true rate from a signal that is mostly noise. Two things follow. The corrected estimate can sit far from the raw count, and the uncertainty around it is wide, because small changes in the precision estimate swing the corrected rate a lot. That is why some entries in Act 6 span ten or more rank positions: the width is the model honestly reporting how little the data pins that entry down.

> **Sidebar — MCMC.** Markov chain Monte Carlo is a way to explore a probability distribution too complex to write down in closed form. It takes a guided random walk through the space of possible answers, spending more time where the answer is more plausible; the collected steps approximate the posterior. We drew 16,000 samples this way (four chains of 4,000, after 2,000 warm-up steps each). The convergence checks sit in the boxes beside the model output.

Three entries (LLM04, LLM08, LLM10) are frame-blind: their incidents come almost entirely from one stratum, so the model cannot cross-check their rates across strata. They stay in the analysis but are flagged, and their rank estimates carry structural uncertainty beyond what the intervals show.

Two measurement gaps widen the intervals further. For 16 of 20 entries the ai-harm recall is not measured directly, so the model uses a conservative prior (roughly 1% recall, a Beta(1, 101)) and corrects upward accordingly. And ai-harm precision is unmeasured, so the model uses a flat Beta(1,1) prior there. Both choices are honest about missing data, and both add width to the posteriors in Act 6.

![Posterior distributions of latent incidence (λ) by entry.](figures/ridge_plot.png){width=70%}

In [ ]:
# Act 5: posterior lambda ridge plot (saves ridge_plot.png @ 300 dpi)
render_ridge_plot(DATA, PREPRINT_FIG)


In [ ]:
# lambda_samples was defined in the Act 5 chart cell (now a library call).
lambda_samples = DATA['lambda_samples']

# Summary statistics from lambda_samples and diagnostic.json
summary_rows = []
for i, eid in enumerate(INFER_ENTRY_ORDER):
    samples = lambda_samples[:, i]
    med = np.median(samples)
    ci_low, ci_high = np.percentile(samples, [5, 95])
    diag_report = DATA['diagnostic']['entry_reports'].get(eid, {})
    flag = diag_report.get('flag', '—')

    summary_rows.append({
        'Entry': eid,
        'Name': ENTRY_NAMES.get(eid, ''),
        '_median_numeric': med,
        'Median λ': f'{med:.4f}',
        '90% CI': f'[{ci_low:.4f}, {ci_high:.4f}]',
        'CI Width': f'{ci_high - ci_low:.4f}',
        'Diagnostic': flag,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('_median_numeric', ascending=False)
summary_df = summary_df.drop(columns=['_median_numeric'])
display(summary_df.style
    .set_caption('Posterior summary: median incident rate, 90% credible interval, diagnostic flag')
    .set_properties(**{'text-align': 'left'})
    .apply(lambda row: ['background: #f0f0f0' if row['Entry'] in FRAME_BLIND else ''
                        for _ in row], axis=1)
)

In [ ]:
# NumPyro model specification sidebar
model_source = (REPO_ROOT / 'engine' / 'model' / 'inference.py').read_text()
# Extract just the model function
model_start = model_source.find('def model(')
model_end = model_source.find('\n    # ------', model_start + 1)
if model_end == -1:
    model_end = model_source.find('\n    try:', model_start)
model_code = model_source[model_start:model_end]

display(sidebar(
    'Deep dive: The NumPyro model specification',
    '<p>This is the actual model we used, written in NumPyro (a probabilistic '
    'programming library for JAX). You do not need to install NumPyro to run this '
    'notebook — the results above are pre-computed.</p>'
    f'<pre style="background: #1a202c; color: #e2e8f0; padding: 1em; '
    f'border-radius: 4px; overflow-x: auto; font-size: 0.85em;">'
    f'{model_code.replace("<", "&lt;").replace(">", "&gt;")}</pre>'
    '<p><strong>Key components:</strong></p>'
    '<ul>'
    '<li><code>lambda</code>: latent prevalence per entry (HalfNormal prior)</li>'
    '<li><code>recall</code>, <code>precision</code>: per-entry, per-stratum '
    'measurement error (Beta priors from gold-set calibration)</li>'
    '<li><code>concentration</code>: over-dispersion parameter (Gamma prior)</li>'
    '<li>The FP leakage term (<code>einsum</code>) accounts for misclassified '
    'incidents that spill from one entry into another</li>'
    '<li>Negative-Binomial likelihood handles count over-dispersion</li>'
    '</ul>'
))

# MCMC diagnostics sidebar
inf_summary = DATA['inference_summary']
max_rhat = max(v for k, v in inf_summary['r_hat'].items() if k.startswith('lambda'))
min_ess = min(v for k, v in inf_summary['ess'].items() if k.startswith('lambda'))
total_draws = inf_summary['num_samples'] * inf_summary['num_chains']

display(sidebar(
    'Deep dive: MCMC convergence diagnostics',
    f'<p>The MCMC sampler ran <strong>{inf_summary["num_chains"]} chains</strong>, '
    f'each with <strong>{inf_summary["num_warmup"]:,}</strong> warmup iterations '
    f'and <strong>{inf_summary["num_samples"]:,}</strong> sampling iterations, '
    f'producing <strong>{total_draws:,}</strong> posterior draws total.</p>'
    '<p><strong>R-hat</strong> measures whether the chains converged to the same '
    'distribution. Values near 1.0 mean convergence. Our maximum R-hat across all '
    f'lambda parameters: <strong>{max_rhat:.6f}</strong> (threshold: ≤1.01). '
    'All parameters pass.</p>'
    f'<p><strong>Effective sample size (ESS)</strong> measures how many independent '
    f'samples the chains produced. Our minimum ESS for lambda parameters: '
    f'<strong>{min_ess:,.0f}</strong> out of {total_draws:,} draws '
    f'({min_ess/total_draws:.0%} efficiency). Higher is better.</p>'
    f'<p><strong>Divergences</strong>: <strong>{inf_summary["divergences"]}</strong>. '
    'Zero divergences means the sampler explored the posterior geometry without '
    'numerical problems. Any non-zero count would indicate regions the sampler '
    'could not traverse reliably.</p>'
))

## Act 6: The incident-derived ranking

This ranking is what the incident data suggests after correcting for classifier error. It is one signal of two, not the final word — the blend in Part I gives it a quarter weight, and Part III shows why that restraint is warranted.

For each entry the model gives a posterior over its true incidence, and we rank entries by their median. Each rank comes with a 90% credible interval.

> **Sidebar — credible interval.** A credible interval is the Bayesian answer to "how sure are we?" A 90% credible interval is the range holding 90% of the posterior's plausible values, so there is a 90% chance the true value sits inside it, given the model and the data. Wide intervals mean low certainty. When an entry's rank interval spans ten positions, the data barely constrains where it belongs.

How to read the chart: each row is an entry, the diamond marks its median rank, and the bar spans the 90% credible interval on that rank. Tight intervals (LLM02 spans roughly 1–6) mean the data constrains the position well. Wide intervals (spanning 6–20) mean the data is compatible with many positions, which happens when precision is low, observations are few, or recall is unmeasured. Grey entries (LLM04, LLM08, LLM10) are frame-blind and carry extra structural uncertainty.

![Incident-derived rank by entry: median rank (diamond) and 90% credible interval (bar).](figures/dumbbell_chart.png){width=68%}

In [ ]:
# Act 6: incident-derived rank dumbbell (saves dumbbell_chart.png @ 300 dpi)
render_dumbbell_chart(DATA, PREPRINT_FIG)


In [ ]:
# (removed) plotly_rankings dropped from the preprint


In [ ]:
# Fill in the incident-rank column from Act 6.  Median incident-derived rank
# per entry = median of per-draw ranks (matches the dumbbell chart).
_lam = DATA['lambda_samples']
_rank_mat = np.argsort(np.argsort(-_lam, axis=1), axis=1) + 1  # 1 = highest rate
_median_rank = {
    eid: float(np.median(_rank_mat[:, i]))
    for i, eid in enumerate(INFER_ENTRY_ORDER)
}

updated_table = pd.DataFrame([
    {
        '#': i + 1,
        'Entry ID': e['entry_id'],
        'Name': e['canonical_name'],
        'Incident Rank': (f"{_median_rank[e['entry_id']]:.0f}"
                          if e['entry_id'] in _median_rank else '\u2014'),
    }
    for i, e in enumerate(DATA['rubric']['entries'])
])
updated_table = updated_table.set_index('#')
display(updated_table.style.set_caption(
    "The table from Act 1, now with incident-derived ranks filled in."
).set_properties(**{'text-align': 'left'}))


## Act 7: Do the experts and the incidents agree?

> **Sidebar — Cohen's κ.** Cohen's kappa (κ) measures agreement between two labelings after subtracting the agreement expected from chance alone. κ = 1 is perfect agreement, κ = 0 is no better than chance, and negative κ is systematic disagreement. The weighted version used here penalizes near-misses less than far-misses. κ comes with an uncertainty interval, and that is the part that matters most: when the interval crosses zero, the data cannot rule out chance-level agreement, so the agreement is weak whatever the point estimate says.

The agreement is weak. Comparing the expert ranking with the incident ranking gives Cohen's weighted κ = 0.20, with a 90% interval of −0.16 to 0.57. The interval crosses zero. We cannot exclude chance-level agreement, and the point estimate of 0.20 sits only in the "slight" band. This is the honest headline of the whole analysis: the incident data agrees with the expert ranking only weakly.

Two things make the interval wide. Only 17 of the 20 entries are measurable (three are frame-blind), and agreement statistics need larger samples to tighten. And the posterior rank distributions are themselves wide, most spanning ten or more positions, which propagates into the agreement estimate.

Five entries disagree the most. Across the joint posterior, the two signals place each of them in different thirds of the ranking more than 83% of the time:

- **LLM01 Prompt Injection:** experts #1 (interval 1–2), incidents #12 (4–18).
- **LLM09 Misinformation:** incidents #2 (1–5), experts #13 (9–16).
- **NEW-MTIE MCP Tool Interface Exploitation:** experts #7 (5–9), incidents #16 (6–20).
- **NEW-PMP Persistent Memory Poisoning:** experts #4 (2–7), incidents #16 (6–20).
- **NEW-WLA Weaponized LLM Abuse:** incidents #8 (3–15), experts #17 (13–20).

![Expert rank against incident rank, entry by entry.](figures/bump_chart.png){width=72%}

![Overlap of the expert and incident rank intervals, per entry.](figures/ci_overlap.png){width=72%}

In [ ]:
# Act 7: expert-vs-incident bump/slope chart (saves bump_chart.png @ 300 dpi)
render_bump_chart(DATA, PREPRINT_FIG)


In [ ]:
# Act 7: rank CI overlap (saves ci_overlap.png @ 300 dpi)
render_ci_overlap(DATA, PREPRINT_FIG)


In [ ]:
display(sidebar(
    'Deep dive: How weighted kappa works',
    '<p>Cohen\'s kappa compares observed agreement to expected agreement by chance. '
    'Weighted kappa extends this to ordinal data — disagreements by 1 tier are '
    'penalized less than disagreements by 2+ tiers.</p>'
    '<p>Concretely: we divide the 20 entries into rank tiers (top 5, 6-10, 11-15, '
    '16-20). Each entry gets a tier from the expert ranking and a tier from the '
    'incident ranking. Kappa measures how often these tiers match, minus what we '
    'would expect from random assignment.</p>'
    '<p>With only <strong>17 measurable entries</strong> (three are frame-blind and excluded), '
    'the sample size is small. Small samples produce wide confidence intervals. '
    'A kappa of 0.20 on 17 observations could easily be consistent with true kappa '
    'values anywhere from -0.16 to 0.57.</p>'
    '<p>For reference: kappa &lt; 0 = worse than chance, 0-0.20 = slight, '
    '0.21-0.40 = fair, 0.41-0.60 = moderate, &gt; 0.60 = substantial.</p>'
))

sb = DATA['selection_bias']
display(sidebar(
    'Deep dive: Selection bias test',
    f'<p>We tested whether incident rates differ systematically between strata '
    f'using a Kruskal-Wallis test (a non-parametric test for differences across groups).</p>'
    f'<p>Result: H = {sb["statistic_value"]:.3f}, p = {sb["p_value"]:.3f}, '
    f'severity = {sb["severity"]}.</p>'
    f'<p>A p-value of {sb["p_value"]:.2f} means we cannot reject the null hypothesis '
    f'that incident rates are similar across strata. In plain language: the security '
    f'and ai-harm strata do not show statistically different patterns of entry prevalence. '
    f'This is reassuring — it means our results are not driven by one stratum dominating.</p>'
))

## Act 8: Where the experts and the incidents disagree

Each of the five mismatches has a plausible cause. Reading them is more useful than averaging them away.

**LLM01 (Prompt Injection): expert #1, incident #12.** Prompt injection is the best-understood LLM attack, and deployed systems defend against it actively. Successful, publicly reported exploits are correspondingly rarer, so the incident record sees fewer of them. Experts rank it first because the attack surface stays enormous even when the defenses mostly hold; the data sees the successes that got through.

**LLM09 (Misinformation): incident #2, expert #13.** The corpus carries a large volume of deepfake and AI-generated disinformation from the AIAAIC harm database. Experts may rank it lower because the category overlaps others (NEW-WLA, ROLL-CMSB) and because many of these incidents describe harm produced by an AI rather than a vulnerability inside an LLM. Act 9B examines that overlap.

Misinformation is the widest disagreement between the two witnesses. The incident record ranks it near the top; the expert vote ranks it near the bottom. The engine's concordance flag puts the probability that the two signals disagree at 99 percent. That number quantifies disagreement between the two signals. It does not quantify how underrated the risk is, and the incident signal behind it rests on the ai-harm stratum, whose precision the corpus does not measure directly. Read it as the entry the incident record most disputes, and the one a better-measured corpus is most likely to move.

**NEW-PMP (Persistent Memory Poisoning) and NEW-MTIE (MCP Tool Interface Exploitation): expert top-5, almost no incidents.** These are emerging threats whose public incident record has not caught up. If the list exists to warn practitioners, expert signal should outweigh incident counts for threats that are new by definition.

**NEW-WLA (Weaponized LLM Abuse): 863 incidents, expert #17.** The large count follows from a broad definition that absorbs AI-generated disinformation, synthetic-media abuse, and deepfake harm. Experts rank it low because most of these describe harm from an AI system rather than an exploitable weakness in an LLM.

![The five largest expert-versus-incident tier mismatches.](figures/paired_dots.png){width=46% wrap=right}

![Frequent keywords in LLM09 (Misinformation) incidents.](figures/theme_bars_llm09.png){width=42% wrap=left}

![Frequent keywords in NEW-WLA (Weaponized LLM Abuse) incidents.](figures/theme_bars_new_wla.png){width=42% wrap=right}

In [ ]:
# Act 8: flagged-entry paired dots (saves paired_dots.png @ 300 dpi)
render_paired_dots(DATA, PREPRINT_FIG)


In [ ]:
# Act 8: incident-theme keyword bars for LLM09 and NEW-WLA (300-dpi PNGs)
render_theme_bars(DATA, PREPRINT_FIG, 'LLM09', 'theme_bars_llm09.png')
render_theme_bars(DATA, PREPRINT_FIG, 'NEW-WLA', 'theme_bars_new_wla.png')


## Act 9: What the data cannot see

### 9A: AI harm without an LLM vulnerability

More than a third of the labeled corpus — 2,394 incidents — landed out of scope, where the models' consensus placed them. These are real AI harms: facial recognition that misidentifies people, hiring tools that discriminate, recommendation engines that radicalize. None describes a vulnerability inside a large language model. They are harms from AI systems, not vulnerabilities of LLMs.

The gap is a property of the sampling frame, not a failure of the taxonomy. The corpus was built by crawling CVE, GHSA, and OSV with AI-related keywords, which pull in anything mentioning "AI" or "machine learning" whether or not an LLM is involved, and AIAAIC covers all AI harms by design. Where an incident sits outside the taxonomy — because it involves non-LLM AI, or describes a societal effect rather than a technical weakness — the incident signal is silent. The out-of-scope pile marks the boundary of what this method can measure.

![Themes among the out-of-scope incidents.](figures/oos_treemap.png){width=85%}

In [ ]:
# Act 9A: out-of-scope theme treemap, plotly static PNG (saves oos_treemap.png @ 300 dpi)
render_oos_treemap(DATA, PREPRINT_FIG)


In [ ]:
# Act 9A: which disagree/split-tier vote combinations the human reviewer sent
# to out-of-scope.  Text summary only (not one of the preprint figures).
prelabel_lookup = {p['incident_id']: p for p in DATA['prelabels']}
disagree_oos_votes = []
for g in DATA['goldset']:
    if (g['llm_consensus'] == 'out-of-scope'
            and g['adjudicated'] == 'accept'
            and g['incident_id'] in prelabel_lookup):
        pl = prelabel_lookup[g['incident_id']]
        if pl['triage_tier'] in ('disagree', 'split'):
            votes = tuple(sorted(v['entry_id'] for v in pl['model_votes']))
            disagree_oos_votes.append(votes)

top_combos = Counter(disagree_oos_votes).most_common(10)
if top_combos:
    print("Disagree/split-tier vote combinations the reviewer sent to out-of-scope:")
    for combo, n in top_combos:
        print(f"  {n:3d}  {' / '.join(combo)}")
else:
    print("No disagree/split-tier incidents were adjudicated to out-of-scope.")


### 9B: The LLM09 / NEW-WLA / ROLL-CMSB confusion boundary

Some categories overlap enough that neither classifiers nor humans can reliably tell them apart. That is a confusion boundary, and it is a property of the categories, not a broken classifier. Three entries share one:

- **LLM09 (Misinformation):** the output is false or misleading.
- **NEW-WLA (Weaponized LLM Abuse):** an adversary uses AI as a weapon.
- **ROLL-CMSB (Cross-Modal Safety Bypass):** the attack runs through image, video, or audio.

A deepfake video spreading political disinformation is all three at once: misleading content, created as a weapon, through a visual modality. The overlap is genuine ambiguity in the taxonomy, not a labeling mistake. So when the incident data ranks LLM09 second, part of that signal comes from incidents that could as easily have been filed under NEW-WLA or ROLL-CMSB. The boundary inflates whichever entry the classifier happens to prefer and deflates the others. The Bayesian model corrects for measured precision, but it cannot correct for ambiguity the human reviewers themselves found hard to resolve.

![Flow of incidents across the LLM09 / NEW-WLA / ROLL-CMSB confusion boundary.](figures/sankey_confusion.png){width=90%}

![Three-by-three confusion among LLM09, NEW-WLA, and ROLL-CMSB.](figures/confusion_matrix_3x3.png){width=42% wrap=right}

In [ ]:
# Act 9B: confusion-boundary Sankey, plotly static PNG (saves sankey_confusion.png @ 300 dpi)
render_sankey_confusion(DATA, PREPRINT_FIG)


In [ ]:
# Act 9B: 3x3 confusion matrix (saves confusion_matrix_3x3.png @ 300 dpi)
render_confusion_matrix_3x3(DATA, PREPRINT_FIG)


In [ ]:
# boundary_list was defined in the Act 9B chart cell (now a library call).
boundary_list = ['LLM09', 'NEW-WLA', 'ROLL-CMSB']

# Show 2-3 real incidents from the confusion boundary
boundary_examples = []
for p in DATA['prelabels']:
    if p['triage_tier'] == 'disagree':
        votes = set(v['entry_id'] for v in p['model_votes'])
        if votes & {'LLM09', 'NEW-WLA', 'ROLL-CMSB'} and len(votes & set(boundary_list)) >= 2:
            boundary_examples.append(p)
    if len(boundary_examples) >= 3:
        break

# Fall back to split tier if not enough disagree examples
if len(boundary_examples) < 2:
    for p in DATA['prelabels']:
        if p['triage_tier'] == 'split':
            votes = set(v['entry_id'] for v in p['model_votes'])
            if len(votes & set(boundary_list)) >= 2:
                boundary_examples.append(p)
        if len(boundary_examples) >= 3:
            break

display(HTML('<h4>Real incidents from the confusion boundary</h4>'))
for ex in boundary_examples[:3]:
    text = html.escape(shorten(ex['text'], width=300, placeholder='...'))
    vote_html = ''.join(
        f'<li><strong>{html.escape(v["model_id"].split("/")[-1])}</strong>: '
        f'{html.escape(v["entry_id"])} ({v["confidence"]:.0%})</li>'
        for v in ex['model_votes']
    )

    # Check if this incident was adjudicated
    adj = next((g for g in DATA['goldset'] if g['incident_id'] == ex['incident_id']), None)
    adj_html = ''
    if adj:
        labels_str = html.escape(", ".join(adj["labels"]) if adj["labels"] else "out-of-scope")
        notes_str = html.escape(adj["notes"]) if adj.get("notes") else ""
        adj_html = (f'<p><strong>Human decision:</strong> {html.escape(adj["adjudicated"])} → '
                    f'{labels_str}'
                    f'{" — " + notes_str if notes_str else ""}</p>')

    display(HTML(
        f'<div style="border: 1px solid #805ad5; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #faf5ff;">'
        f'<code>{html.escape(ex["incident_id"])}</code> · tier: '
        f'{html.escape(ex["triage_tier"])}<br>'
        f'<p style="color: #333; font-style: italic;">{text}</p>'
        f'<ul style="margin: 0.5em 0;">{vote_html}</ul>'
        f'{adj_html}'
        f'</div>'
    ))

## Act 10: What Part II shows

Where the data and the experts agree, confidence is highest. LLM02 (Sensitive Information Disclosure) sits near the top of both. ROLL-SICG, NEW-ITSCD, and NEW-MSDA sit near the bottom of both. These positions hold across the uncertainty ranges, and they are the safest to act on.

Where the data pushes back, it does so for readable reasons. LLM09's incident volume far exceeds its expert rank, driven partly by a broad definition that sweeps in AI-adjacent harms and inflated by the LLM09 / NEW-WLA / ROLL-CMSB confusion boundary, which makes all three counts less reliable than entries with cleaner definitions. NEW-WLA shows the same pattern.

Where the experts see what the incidents miss, the gap is about timing. NEW-PMP and NEW-MTIE have strong expert signal and almost no public incidents, because the threats are new. For emerging risks the expert signal leads, which is exactly what a quarter-weight on the data is meant to allow.

The method triangulates two imperfect signals; it does not hand down a verdict from either. The incident data has structural biases: a sampling frame that misses unreported incidents, measured classifier error, and taxonomy-frame circularity, which means we are partly measuring the classifier's preferences rather than the true threat distribution. The expert vote has its own biases: availability, recency, anchoring to last year's list. The value is in the comparison. Where they agree, confidence rises; where they diverge, the divergence itself is the finding.

Stated plainly: the incident data agrees with the expert ranking only weakly (Cohen's κ ≈ 0.20, with an interval that crosses zero), and — as Part III shows next — the expert ranking is robust: four frontier classifiers and a ground-truth check do not move it. Weak agreement and a stable ranking are not in tension. A quarter-weight corrective that agrees weakly is doing exactly what it was designed to do, and a ranking a stronger classifier cannot improve is one to rely on while the measurement gets better.

# Part III — Robustness Under Frontier Classifiers

## Act 11: Does a better classifier change the ranking?

The incident-derived ranking depends on the classifier that labeled the corpus. If a stronger model would reorder the list, the ranking is an artifact of a weak classifier and should not be trusted. So we pre-registered a bake-off: four frontier models re-labeled the evaluation set, and the rule for declaring a winner was fixed before any model ran. A model had to beat the 2026 incidence floor on balanced accuracy and clear a significance test.

> **Sidebar — balanced accuracy.** Plain accuracy flatters a classifier on lopsided data: a model that always guesses the common class scores high while missing every rare one. Balanced accuracy averages the recall within each class, so a rare category counts as much as a common one. On this kind of task it runs from about 0.5 (chance) to 1.0 (perfect).

The bake-off returned no winner. The 2026 floor scored balanced accuracy 0.863. All four frontier models scored below it: llama-405b 0.744, qwen3-235b 0.733, deepseek-v3 0.711, mistral-large-2411 0.691. None cleared the floor, so under the pre-registration the 2026 ranking stands. (Every number in this section is read from `cycles/2026-rarr/results/robustness_validation.json` and the RARR conclusion beside it.)

A null result, no model beating the floor, could mean the floor is genuinely good or that the test is weak. So we checked the floor against ground truth directly. Scored against the adjudicated gold set as truth, the floor's incidence ranking correlates with the true ranking at Spearman ρ = 0.918.

> **Sidebar — Spearman ρ.** Spearman's rho measures how well two rankings agree on order, ignoring the exact scores. It runs from −1 (one ranking is the reverse of the other) through 0 (no relationship) to +1 (identical order). A ρ of 0.918 means the floor's ordering is very close to the true ordering.

Every frontier model's apparent edge over the floor dissolves under a bootstrap.

> **Sidebar — bootstrap.** The bootstrap estimates how much a number would wobble if the data had come out slightly differently. It re-samples the observations with replacement many times, recomputes the number on each re-sample, and reads the spread as a confidence interval. When that interval crosses zero, the apparent effect is within noise.

The paired-bootstrap interval for each model's ranking-fidelity gain over the floor crosses zero: llama-405b +0.001 (95% interval −0.047 to +0.055), the ensemble +0.009 (−0.032 to +0.050), mistral-large-2411 −0.012 (−0.053 to +0.028). Reweighting the gold set to the corpus's class mix raises the floor to ρ = 0.971 and leaves the best frontier configuration statistically tied (interval −0.025 to +0.024). The figure shows each classifier's ranking fidelity against the floor.

![Ranking fidelity (Spearman ρ against held-out truth) for each frontier classifier and the ensemble, against the 2026 incidence floor. Every frontier gain over the floor has a bootstrap interval that crosses zero.](figures/rarr_robustness.png){width=48% wrap=left}

### The recall correction, and its one honest limit

One difference between the floor and the frontier models survives correction. The floor classifier over-files incidents into a few crowded categories (LLM02, LLM09, ROLL-CMSB), which inflates their raw counts. The pipeline's recall-and-precision correction is built to remove exactly this kind of distributional gap, and it does: after correction, the floor and the ensemble reach effectively the same per-class magnitude accuracy (cross-validation-corrected neg-L2 −0.0024 for the floor, −0.0033 for the ensemble; the ensemble-minus-floor difference has a 95% interval of −0.0009 to +0.0007, which crosses zero).

The correction has a limit it cannot cross. The 2026 classifier never predicts "out of scope": it assigns a specific taxonomy category to every incident, including the roughly 38% of the gold set that is truly out of scope. Its out-of-scope recall is therefore 0%, against 0.49 for a frontier model on the same set. A recall correction adjusts for how often a classifier misses a class it does predict; it cannot recover a class the classifier never predicts at all. On this one axis the frontier models are genuinely better: they spot out-of-scope incidents that the floor files into a category. This changes the magnitudes of a few per-class counts. It does not change the rank order: the reweighting and bootstrap tests above already price in the false-positive inflation, and the ordering holds.

The conclusion of Part III is narrow and firm. No frontier classifier reorders the 2026 ranking, on either the ordinal order or the recall-corrected magnitudes. The ranking is robust to classifier choice. The one measured advantage of the frontier models, out-of-scope detection, changes magnitudes the pipeline already corrects for, not the order of the list.

In [ ]:
# Act 11: RARR robustness — ranking fidelity (Spearman rho vs held-out truth)
# for each frontier classifier and the ensemble, against the incidence floor.
# Saves rarr_robustness.png @ 300 dpi.
_rarr_robustness = json.loads(
    (RARR_CYCLE / 'results' / 'robustness_validation.json').read_text()
)
render_rarr_robustness(_rarr_robustness, PREPRINT_FIG)


## The 2026 blended Top 10

The blend combines two witnesses into a distribution over each risk's final position. The expert vote carries three-quarters weight, the incident data one quarter. Each risk gets a spread of plausible positions, and the ten fall into three tiers.

> **Sidebar — distribution over positions.** Each posterior draw pairs one sample of the incident rates with one sample of the expert ranking, blends them, and reads off a position. Sixteen thousand draws give a distribution over positions for each risk, so a firm placement and a coin flip look different on the page.

![Each risk at its mean position, with a bar spanning the 5th to 95th percentile. Three tiers read directly: a tight pair, an overlapping band, and a wide tail.](figures/blend_position_intervals.png){width=70% wrap=left}

**The co-leading pair.** Sensitive Information Disclosure and Prompt Injection hold the top, each a near-certain top-three risk (P(top-3) 0.99 and 0.95). The two blends disagree on which ranks first: the simpler rank-space blend puts Sensitive Information Disclosure ahead, the probabilistic blend puts Prompt Injection ahead, and their intervals overlap. We report them as co-leading, with no method-independent first place.

**The tied band.** Excessive Agency, Supply Chain, and Data and Model Poisoning form a middle band with overlapping intervals. Excessive Agency moves up into the band from its published position, the one clear mover here.

**The tail.** Unbounded Consumption sits at the top of the tail, placed by the expert vote alone (its incident recall the corpus cannot estimate), and it reaches the top five in about a third of the draws. The remaining four — Misinformation, Hidden Context Exposure, Vector and Embedding Weaknesses, Improper Output Handling — reach the top five in under one draw in twenty. We present the tail as a group and do not report an order inside it.

> **Sidebar — why a distribution beats a single rank number.** A single rank hides its own uncertainty. Two adjacent tail positions differ by a fraction of a place across the draws, so a printed rank would claim precision the data does not carry. The tiers report only what the spread supports.

A simpler rank-space blend, which uses only the order of each witness and discards the magnitudes, gives nearly the same bulk ordering (Kendall's tau computed at run time, about 0.87). The one place the two methods disagree is the top, which is why we present the top two as a pair.

In [ ]:
# The 2026 candidate ballot: 10 incumbents (LLM01-LLM10) + 6 NEW-* + 4 ROLL-*,
# with arrows from each roll-up to the incumbent it folds into.
# Saves entry_expansion_map.png @ 300 dpi.
_entries = load_entries(CYCLE / 'taxonomy' / 'taxonomy.json')
render_entry_expansion_map(_entries, PREPRINT_FIG)


In [ ]:
# The 2026 blended Top 10 (probabilistic blend). Computed by engine.decide.blend from the
# committed posterior samples; the rank-space order (existing engine) and the Kendall tau
# between the two are computed here so the prose cites no hand-typed number.
from engine.decide.blend import blend, load_inputs
from engine.report.blend_2025_2026 import blended_ranking, load_entries
from scipy.stats import kendalltau

BLEND_MANIFEST = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026' / 'blend' / 'blend_manifest.json'
blend_result = blend(load_inputs(BLEND_MANIFEST))

# rank-space order from the existing tested engine (for the robustness-lens tau).
_rank_md = DATA['rank_comparison_md']
_lam, _vote = {}, {}
for _line in _rank_md.splitlines():
    if _line.startswith('|') and 'Entry' not in _line and not _line.startswith('|--'):
        _c = [c.strip() for c in _line.split('|')[1:-1]]
        if len(_c) >= 3 and re.match(r'([\d.]+)', _c[1]) and re.match(r'([\d.]+)', _c[2]):
            _lam[_c[0]] = float(re.match(r'([\d.]+)', _c[1]).group(1))
            _vote[_c[0]] = float(re.match(r'([\d.]+)', _c[2]).group(1))
_ent = load_entries(CYCLE / 'taxonomy' / 'taxonomy.json')
_inc = [e['entry_id'] for e in _ent if e['group'] == 'incumbent']
_child = {e['entry_id']: e['rolled_into'] for e in _ent if e['group'] == 'rollup'}
_fl, _fv = dict(_lam), dict(_vote)
for _c2, _p in _child.items():
    if _p in _fl and _c2 in _lam:
        _fl[_p] = min(_fl[_p], _lam[_c2]); _fv[_p] = min(_fv[_p], _vote[_c2])
_rankspace = [b['entry_id'] for b in blended_ranking({e: _fv[e] for e in _inc}, {e: _fl[e] for e in _inc}, 0.75)]
_prob_rank = {e: i for i, e in enumerate(blend_result.order)}
_rs_rank = {e: i for i, e in enumerate(_rankspace)}
blend_tau = float(kendalltau([_prob_rank[e] for e in _inc], [_rs_rank[e] for e in _inc]).statistic)

print('Tiers:', blend_result.tiers)
print(f'Kendall tau (probabilistic vs rank-space): {blend_tau:.2f}')
for e in blend_result.order:
    print(f"  {e:6s} pos={blend_result.mean_position[e]:.2f} "
          f"P3={blend_result.p_top3[e]:.2f} P5={blend_result.p_top5[e]:.2f} CI={blend_result.interval[e]}")


In [ ]:
# blend_position_intervals.png — each risk at its mean position (dot) with a 5-95 pct bar,
# grouped into the three tiers. Replaces the retired rank-change slopegraph.
import matplotlib.pyplot as plt
_tier_of = {e: t for t, es in blend_result.tiers.items() for e in es}
_tc = {'pair': '#1b6ca8', 'band': '#f2a154', 'tail': '#9aa0a6'}
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for row, e in enumerate(blend_result.order):
    lo, hi = blend_result.interval[e]
    ax.plot([lo, hi], [row, row], color=_tc[_tier_of[e]], lw=6, solid_capstyle='round', alpha=0.55)
    ax.plot(blend_result.mean_position[e], row, 'o', color=_tc[_tier_of[e]], ms=8)
    ax.text(0.3, row, ENTRY_NAMES.get(e, e), va='center', ha='right', fontsize=9)
ax.set_yticks([]); ax.invert_yaxis()
ax.set_xlabel('Position among the ten (1 = highest priority); bar = 5th-95th percentile')
ax.set_xlim(0.3, 10.7); ax.set_xticks(range(1, 11))
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(PREPRINT_FIG / 'blend_position_intervals.png', dpi=300, bbox_inches='tight'); plt.show()


In [ ]:
# blend_top_k_probs.png — grouped bars of P(top-3) and P(top-5) per risk, ordered by tier.
import numpy as np
_ord = list(blend_result.order); _x = np.arange(len(_ord)); _w = 0.38
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.bar(_x - _w/2, [blend_result.p_top3[e] for e in _ord], _w, label='P(top 3)', color='#1b6ca8')
ax.bar(_x + _w/2, [blend_result.p_top5[e] for e in _ord], _w, label='P(top 5)', color='#f2a154')
ax.set_xticks(_x); ax.set_xticklabels([ENTRY_NAMES.get(e, e) for e in _ord], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Probability'); ax.set_ylim(0, 1.02); ax.legend(frameon=False)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(PREPRINT_FIG / 'blend_top_k_probs.png', dpi=300, bbox_inches='tight'); plt.show()


In [ ]:
# Consistency check: every published ranking number must equal the engine result.
# The build executes this cell, so a prose number that drifts fails the build.
import json
_g = json.loads((REPO_ROOT / 'projects/owasp-llm/cycles/2026/blend/blend_golden.json').read_text())
assert list(blend_result.order) == _g['order'], "order drift vs golden"
assert blend_result.tiers['pair'] == ('LLM01', 'LLM02')
assert blend_result.tiers['band'] == ('LLM06', 'LLM03', 'LLM04')
assert blend_result.tiers['tail'] == ('LLM10', 'LLM09', 'LLM07', 'LLM08', 'LLM05')
assert abs(blend_result.p_top3['LLM01'] - 0.99) < 0.02
assert abs(blend_result.p_top3['LLM02'] - 0.95) < 0.03
assert abs(blend_result.p_top5['LLM10'] - 0.33) < 0.05      # borderline tail, cited in prose
assert all(blend_result.p_top5[e] < 0.05 for e in ('LLM09', 'LLM07', 'LLM08', 'LLM05'))
assert 0.80 <= blend_tau <= 0.95                            # rank-space robustness lens
_rob = json.loads((RARR_CYCLE / 'results' / 'robustness_validation.json').read_text())
assert _rob['ranking_fidelity_spearman_vs_truth']['floor'] > 0.85
_bl = json.loads((BASELINES / 'rankings_baselines.json').read_text())
assert _bl['previous_ranking']['kappa_median'] == 0.2028985507246377
print("consistency OK")


## Glossary

Every term defined in a sidebar, collected here for reference.

**0.75 / 0.25 blend.** The rule that combines the expert vote and the incident data into one ranking. Each risk's expert rank and incident rate are placed on a common scale (a z-score) and combined at fixed weights, three-quarters vote and one quarter data, once per posterior draw. Sorting the resulting scores highest-first for each draw produces a distribution over positions for every risk, summarized here as three tiers.

**Balanced accuracy.** Accuracy that averages the recall within each class, so a rare category counts as much as a common one. It avoids the trap of plain accuracy, which a model can inflate by always guessing the common class. On this task it runs from about 0.5 (chance) to 1.0 (perfect).

**Blind labeling.** Recording a human judgment before seeing the machine's answer, so the human is not anchored to it. The gold-set reviewer labeled each incident from its text alone, then revealed the model votes, then decided.

**Bootstrap.** A way to gauge how much a number would wobble under slightly different data. Re-sample the observations with replacement many times, recompute the number each time, and read the spread as a confidence interval. An interval that crosses zero means the apparent effect is within noise.

**Classifier.** Any procedure that reads an input and assigns it to one of a fixed set of categories. Ours is an ensemble of three language models voting on which taxonomy entry (or "out of scope") each incident belongs to. It is a measuring instrument, not ground truth.

**Cohen's κ (kappa).** Agreement between two labelings after subtracting the agreement expected by chance. κ = 1 is perfect, κ = 0 is chance-level, negative κ is systematic disagreement. When its uncertainty interval crosses zero, the agreement is weak whatever the point estimate.

**Credible interval.** The Bayesian range of plausible values for a quantity. A 90% credible interval holds 90% of the posterior's probability, so the true value sits inside it with 90% probability given the model and data. Wide means uncertain.

**Credible interval over a rank.** A credible interval computed on an entry's rank position, the same idea used elsewhere in this report for a count-scale quantity like λ. It gives the middle span of plausible ranks a risk could hold under the model and the data. The interval assumes a single correct rank is the value it brackets, an assumption that holds less firmly for the three frame-blind entries, whose rank comes from the vote alone.

**Distribution over positions.** The spread of positions a risk can land at across many posterior draws. Each draw pairs one incident-rate sample with one expert-rank sample, blends them, and places the risk at a position. Collecting the position from all 16,000 draws produces a full spread for each risk, from a tight cluster near one place to a wide scatter across many.

**Frame-blind drop.** The adjustment applied to the three frame-blind entries (Data and Model Poisoning, Vector and Embedding Weaknesses, Unbounded Consumption): their weight shifts from three-quarters vote and one quarter data to full vote weight, so the blend places them from the expert rank alone. Their incident rates still enter the shared scale used to score every other risk. The shift changes where a frame-blind entry lands, and the report states that plainly.

**Gold set.** A batch of records labeled carefully by a human to serve as the answer key — here, 1,200 adjudicated incidents used both to calibrate the classifier and as the ground truth for the robustness check.

**Incident corpus.** A fixed, documented collection of incident records assembled for analysis. Ours is a dated snapshot of publicly reported LLM-security incidents, each a short text description, drawn from public databases.

**Kendall's tau (τ).** A second measure of how well two rankings agree on order, alongside Spearman's ρ. It checks every pair of entries and counts how often the two rankings place them in the same relative order, turning that count into a score from −1 (every pair reversed) through 0 (no relationship) to +1 (every pair agrees). This report uses it to compare the probabilistic blend's bulk order against a simpler rank-space blend that keeps only the ranks and discards the underlying scores.

**Latent incidence (λ).** The true, unobserved underlying rate at which incidents of a category occur, as distinct from the raw count the classifier reported. The model infers λ from the noisy counts after correcting for precision and recall.

**MCMC (Markov chain Monte Carlo).** A method for exploring a probability distribution too complex to write down in closed form, by a guided random walk that spends more time where the answer is more plausible. The collected steps approximate the posterior; we drew 16,000.

**Negative-binomial measurement-error model.** A count model with two features: it treats classifier counts as noisy measurements of the true rate and corrects for the noise (measurement-error), and it uses the negative-binomial distribution, which — unlike the Poisson — lets the spread of counts exceed their average (over-dispersion).

**Out-of-scope.** The category for incidents that belong to none of the 20 taxonomy entries: a real AI harm that is not a vulnerability in a large language model. Marking an incident out of scope is a correct classification.

**P(top-k).** Across the 16,000 posterior draws, the share that place a risk within the top k positions (k = 3 or 5 in this report). A risk that is near-certain to rank highly shows a P(top-3) close to 1. The three-quarters vote weight dominates this probability wherever the vote and the data point the same direction, so a high P(top-k) usually traces back to a strong expert-vote placement that the incident data does not contradict.

**Precision.** Of the incidents the classifier files under a category, the fraction that truly belong there. Low precision means most of what lands in a category does not belong to it.

**Prior and posterior.** The prior is what the model assumes before seeing this data; the posterior is the updated belief after combining prior and data. The posterior is a distribution of plausible values, not a single number.

**Probabilistic blend.** The method this report uses for the 2026 ranking: combine the incident-rate posterior and the expert-vote posterior in score space, once per posterior draw, and read off a position for each risk. Repeating this over 16,000 draws produces a distribution over positions for each risk. The report summarizes that distribution as a mean position, a P(top-k), a credible interval over each rank, and three tiers.

**Recall.** Of the incidents that truly belong to a category, the fraction the classifier finds. A classifier can have high precision and low recall, or the reverse; the two are measured separately.

**Spearman ρ (rho).** How well two rankings agree on order, ignoring exact scores. It runs from −1 (reversed) through 0 (unrelated) to +1 (identical order).

**Sum-prevalence fold.** The rule for combining a rolled-up child entry's incident rate into its parent on the data axis: the child's rate adds to the parent's, so incidents recorded against the child still count toward the parent's total. Four entries fold this way in the 2026 blend. The vote axis folds by a separate rule: the parent keeps the better of its own rank and its child's rank, since votes do not add the way rates do.

**Tied tier.** A group of entries whose plausible-position intervals overlap enough to fold into one block for reporting purposes. The 2026 blend sorts the ten incumbents into three such tiers: a co-leading pair at the top, a tied band in the middle, and a wide tail with no defended order inside it. Membership in a tier comes from the posterior draws themselves, a data-driven grouping.

## Limitations and Independent-Review Status

This is an internally rigorous analysis with real limits. We state them plainly rather than bury them.

**The gold set is single-author.** One reviewer (the project author) adjudicated all 1,200 gold-set incidents that calibrate the classifier and serve as the ground truth in Part III. The reviewer's independent blind label disagreed with the model consensus at a rate of 0.75, and the final adjudication overrode the consensus on 553 of the 1,200 incidents. Those rates are high, and they reflect genuinely ambiguous categories rather than a broken pipeline. A single annotator cannot measure inter-rater reliability; a second independent adjudicator would harden the truth target. The ranking-fidelity bootstrap median of exactly 0.000 makes a hidden improvement unlikely regardless, but the single-author gold set remains the central limitation.

**The reviewers are interim.** The rubric reviewer and the statistical reviewer for this analysis are, at this stage, the ranking author. Independent adjudication of the gold set and independent statistical review are future work, not completed steps. Read the findings as exploratory and pending independent adjudication.

**The agreement signal is weak and uncertain.** Cohen's κ between the vote and the incident data is 0.20 with a 90% interval of −0.16 to 0.57. The interval crosses zero, so we cannot rule out that the two rankings agree only by chance. The honest bottom line: weak agreement, not confirmation.

**The corpus has known blind spots.**

- *Stratum imbalance.* The labeled corpus is 6,297 security incidents against 342 ai-harm incidents. Precision was hand-verified only on the security stratum; ai-harm precision falls back to an uninformative prior. Conclusions about ai-harm categories rest on weaker measurement.
- *The out-of-scope blind spot.* The base classifier never predicts "out of scope" and files every incident into some category, including the roughly 38% of the gold set that belongs in none. Part III shows the ranking survives the resulting false-positive inflation, but the raw per-class counts overstate the crowded categories.
- *Frame-blind entries.* Three entries — LLM04, LLM08, LLM10 — draw their signal from a single stratum, so the corpus cannot cross-check their recall, and their data ranks are soft by construction. Several broad categories (LLM09, NEW-WLA, ROLL-CMSB) also sit on confusion boundaries the classifier cannot cleanly resolve, so their counts are less reliable than entries with sharp definitions.

**Status.** This report is exploratory and internally rigorous. It is not peer-reviewed, and it is not the official OWASP release. Full external publication would require an independent adjudicated gold set, independent statistical review, and a broader corpus that narrows the κ interval. Until then the blended ranking is a working reconciliation of two imperfect signals, offered for scrutiny.

## Scope and Authority

This report is an incident-data analysis authored by two members of the OWASP working group. It is not the official OWASP Top 10 for LLM Applications, it does not supersede the official list or the process that produces it, and it does not speak for OWASP.

"The 2026 list" here means the working group's expert-driven candidate ranking. This report stress-tests that ranking against incident data; it does not set it. Where the blend reorders the incumbents, that reordering is an analytical result for discussion, not a change to any published list.

The work is exploratory and internally rigorous, not a peer-reviewed finding. Its value is transparency and a robustness check: it shows how a community-expert ranking holds up when confronted with a large-scale incident corpus, and it is candid about where the data is weak. Read it as a stress test offered for scrutiny, not as an authority on the ranking.